# EarthScape Climate Agency — Notebook 05: PySpark Big Data Processing

### Objective:
Demonstrate distributed large-scale data processing with Apache Spark (PySpark), including SparkSession initialization, schema inference, distributed transformations, multi-dimensional aggregations, and Parquet storage.


## 1. Initialize PySpark SparkSession


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, year, month, count, avg, sum as spark_sum

spark = SparkSession.builder \
    .appName("EarthScape_Climate_BigData") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"SparkSession Active: Version {spark.version}")


## 2. Load Dataset into Spark DataFrame & Inspect Schema


In [ ]:
CSV_PATH = "../WeatherEvents_Jan2016-Dec2022.csv"
spark_df = spark.read.option("header", "true").option("inferSchema", "true").csv(CSV_PATH)
spark_df.printSchema()


## 3. Distributed Transformations & Timestamp Parsing


In [ ]:
df_transformed = spark_df \
    .withColumn("StartTime_Parsed", to_timestamp(col("StartTime(UTC)"))) \
    .withColumn("EndTime_Parsed", to_timestamp(col("EndTime(UTC)"))) \
    .withColumn("Year", year(col("StartTime_Parsed"))) \
    .withColumn("Month", month(col("StartTime_Parsed"))) \
    .filter(col("LocationLat").isNotNull() & col("LocationLng").isNotNull())

print("Transformed sample:")
df_transformed.select("EventId", "Type", "Severity", "Year", "Month", "State").show(5)


## 4. Distributed Aggregations: Events by State


In [ ]:
state_agg = df_transformed.groupBy("State") \
    .agg(count("EventId").alias("TotalEvents"), avg("Precipitation(in)").alias("AvgPrecipitation")) \
    .orderBy(col("TotalEvents").desc())

state_agg.show(10)


## 5. Distributed Aggregations: Events by Type & Severity


In [ ]:
type_sev_agg = df_transformed.groupBy("Type", "Severity") \
    .agg(count("EventId").alias("Count")) \
    .orderBy(col("Count").desc())

type_sev_agg.show(15)


## 6. Distributed Yearly & Monthly Aggregations


In [ ]:
yearly_spark = df_transformed.groupBy("Year").count().orderBy("Year")
yearly_spark.show()


## 7. Export Processed Data to Optimized Parquet


In [ ]:
output_parquet = "../data/processed/pyspark_climate_summary.parquet"
df_transformed.sample(False, 0.05).write.mode("overwrite").parquet(output_parquet)
print(f"Saved PySpark processed summary to {output_parquet}")


### Conclusion:
PySpark efficiently executes distributed transformations, schema enforcement, multi-dimensional aggregations, and high-performance Parquet storage for big climate datasets.
